<a href="https://colab.research.google.com/github/melatisurya/Tugas_UAS_DeepLearning_Sepaket/blob/main/M3_Sepaket_T1C_SemuaRun_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M3 · Tim Sepaket · T1-C · Semua run dalam satu notebook
Satu notebook untuk **R2, R4, R5, R6 → R7 (3 seed) → R8 (set uji) → galeri kesalahan + peta aktivasi**. Jalankan **Runtime → Jalankan semua**, biarkan tab terbuka.

**Aman terputus:** setiap run yang selesai ditandai di Google Drive. Kalau Colab terputus, jalankan ulang dari sel paling atas; run yang sudah selesai dilewati.

**Estimasi waktu (T4):** R1 dan R3 masing-masing ±0,2 jam (terukur). R5 (ResNet-50) lebih berat, perkiraan ±1 jam (belum terukur). Total kira-kira 3–5 jam. Kalau tenggat mepet, jalankan bertahap dan pakai hasil yang sudah ada.

**Catatan metodologi:**
- Semua run dievaluasi ulang di sini dengan fungsi yang sama dan **grid ambang confidence diperluas (0,25–0,7)**, termasuk R1 dan R3 dari bobot tersimpan.
- R7 = dua konfigurasi terbaik di validasi, **satu per paradigma** (satu YOLO, satu Faster R-CNN), masing-masing 3 seed (42, 1, 2).
- **Versi 2:** grid ambang confidence diperluas sampai 0,9 (di versi 1 semua run memilih 0,7 = batas atas), R1 dilatih ulang karena bobot lamanya tidak ditemukan, dan jumlah seed R7 bisa diatur lewat `SEEDS_EXTRA`.
- Set uji hanya dibaca di bagian R8, satu kali per run (hasilnya disimpan dan tidak dihitung ulang).

In [ ]:
!pip -q install ultralytics torchmetrics pycocotools openpyxl
import os, glob, json, random, time, copy, traceback, platform
import numpy as np, pandas as pd, torch, torchvision
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection import MeanAveragePrecision
from ultralytics import YOLO

NAMES = ['WBC', 'RBC', 'Platelets']
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'GPU tidak aktif: Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Sepaket_T1C'          # folder yang sama dengan notebook sebelumnya
for d in ['runs', 'results', 'figures']: os.makedirs(f'{OUT}/{d}', exist_ok=True)
if not os.path.exists('/content/TXL-PBC_Dataset'):
    !git clone --depth 1 https://github.com/lugan113/TXL-PBC_Dataset.git /content/TXL-PBC_Dataset
ROOT = '/content/TXL-PBC_Dataset/TXL-PBC'
YAML = f'{OUT}/txl_pbc.yaml'
open(YAML, 'w').write(f"path: {ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 3\nnames: {NAMES}\n")

## 1. Daftar run dan status

In [ ]:
# Konfigurasi semua run. {seed} diisi otomatis.
RUNS = {
 'R1': dict(kind='yolo',  w='yolov8n.pt', imgsz=640, freeze=None, name='R1_yolov8n_640_s{seed}'),
 'R2': dict(kind='yolo',  w='yolov8s.pt', imgsz=640, freeze=None, name='R2_yolov8s_640_s{seed}'),
 'R6': dict(kind='yolo',  w='yolov8n.pt', imgsz=640, freeze=10,   name='R6_yolov8n_640_freeze_s{seed}'),
 'R3': dict(kind='frcnn', arch='mobilenet', imgsz=640, name='R3_frcnn_mobilenet_640_s{seed}'),
 'R4': dict(kind='frcnn', arch='mobilenet', imgsz=416, name='R4_frcnn_mobilenet_416_s{seed}'),
 'R5': dict(kind='frcnn', arch='resnet50',  imgsz=640, name='R5_frcnn_resnet50_640_s{seed}'),
}
def run_name(rid, seed): return RUNS[rid]['name'].format(seed=seed)
def ckpt_path(rid, seed):
    n = run_name(rid, seed)
    return f'{OUT}/runs/{n}/weights/best.pt' if RUNS[rid]['kind'] == 'yolo' else f'{OUT}/runs/{n}/best.pt'

# Run yang SUDAH selesai dilatih di notebook sebelumnya (dilewati). Tambahkan nama run bila R2/R4/R5/R6 sudah selesai di notebook lain.
ALREADY_DONE = {'R3_frcnn_mobilenet_640_s42'}      # R1 dilatih ulang di bawah (bobot R1 lama tidak ditemukan di Drive)
LEGACY_GPU_H = {'R1_yolov8n_640_s42': 0.198, 'R3_frcnn_mobilenet_640_s42': 0.211}

def is_done(rid, seed):
    n = run_name(rid, seed)
    return os.path.exists(f'{OUT}/runs/{n}/done.json') or (n in ALREADY_DONE and os.path.exists(ckpt_path(rid, seed)))
for rid in RUNS: print(rid, run_name(rid, 42), '-> selesai' if is_done(rid, 42) else '-> belum')

## 2. Data, evaluasi, dan model (fungsi bersama)

In [ ]:
def read_gt(img_path):
    W, H = Image.open(img_path).size
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    boxes, labels = [], []
    if os.path.exists(lp):
        for ln in open(lp):
            v = ln.split()
            if len(v) < 5: continue
            c = int(v[0]); cx, cy, w, h = map(float, v[1:5])
            boxes.append([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H]); labels.append(c)
    return np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=int)

def load_split(split):
    files = sorted(glob.glob(f'{ROOT}/images/{split}/*')); gts = []
    for f in files:
        b, l = read_gt(f)
        gts.append(dict(boxes=torch.tensor(b).reshape(-1, 4), labels=torch.tensor(l, dtype=torch.long)))
    return files, gts
SPLITS = {s: load_split(s) for s in ['val', 'test']}

class TXL(torch.utils.data.Dataset):
    def __init__(self, split, train=False):
        self.files = sorted(glob.glob(f'{ROOT}/images/{split}/*')); self.train = train
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        p = self.files[i]; img = Image.open(p).convert('RGB'); W, H = img.size
        b, l = read_gt(p); t = to_tensor(img); b = torch.tensor(b); l = torch.tensor(l + 1, dtype=torch.int64)
        if self.train:
            if random.random() < 0.5: t = t.flip(-1); b[:, [0, 2]] = W - b[:, [2, 0]]
            if random.random() < 0.5: t = t.flip(-2); b[:, [1, 3]] = H - b[:, [3, 1]]
        return t, {'boxes': b, 'labels': l}
def collate(b): return tuple(zip(*b))

NMS_IOU = 0.7
def build_model(arch, size):
    kw = dict(weights='DEFAULT', min_size=size, max_size=size, box_score_thresh=0.001, box_nms_thresh=NMS_IOU, box_detections_per_img=100)
    m = fasterrcnn_mobilenet_v3_large_fpn(**kw) if arch == 'mobilenet' else fasterrcnn_resnet50_fpn(**kw)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, len(NAMES) + 1)
    return m

@torch.no_grad()
def predict_torch(model, files, bs=8):
    model.eval(); preds = []
    for i in range(0, len(files), bs):
        ims = [to_tensor(Image.open(f).convert('RGB')).to(DEV) for f in files[i:i+bs]]
        for o in model(ims):
            preds.append(dict(boxes=o['boxes'].cpu(), scores=o['scores'].cpu(), labels=(o['labels'] - 1).cpu()))
    return preds

def predict_yolo(model, files, imgsz):
    preds = []
    for r in model.predict(files, conf=0.001, imgsz=imgsz, iou=NMS_IOU, verbose=False, stream=True, device=0):
        b = r.boxes; preds.append(dict(boxes=b.xyxy.cpu(), scores=b.conf.cpu(), labels=b.cls.cpu().long()))
    return preds

def load_model(rid, seed):
    cfg = RUNS[rid]
    if cfg['kind'] == 'yolo': return YOLO(ckpt_path(rid, seed))
    m = build_model(cfg['arch'], cfg['imgsz']); m.load_state_dict(torch.load(ckpt_path(rid, seed), map_location=DEV)); return m.to(DEV).eval()

def predict(rid, model, files):
    cfg = RUNS[rid]
    return predict_yolo(model, files, cfg['imgsz']) if cfg['kind'] == 'yolo' else predict_torch(model, files)

def eval_map(preds, gts):
    mt = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=True)
    mt.update(preds, gts); r = mt.compute()
    return dict(mAP50=float(r['map_50']), mAP50_95=float(r['map']), AP_small=float(r['map_small']),
                AP_WBC=float(r['map_per_class'][0]), AP_RBC=float(r['map_per_class'][1]), AP_Platelets=float(r['map_per_class'][2]))

def count_metrics(preds, gts, thr):
    out = {}
    for c, n in enumerate(NAMES):
        gg = np.array([(d['labels'] == c).sum().item() for d in gts])
        pp = np.array([((d['labels'] == c) & (d['scores'] >= thr)).sum().item() for d in preds])
        nz = gg > 0
        out[n] = dict(MAE=float(np.abs(pp - gg).mean()),
                      MAPE=float((np.abs(pp[nz] - gg[nz]) / gg[nz]).mean() * 100) if nz.any() else float('nan'))
    return out
THR_GRID = [0.25, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
RES_TAG = 'g2'      # versi grid ambang; hasil lama (grid sempit) diabaikan

def latency_cpu(rid, seed, files, n=50, warmup=5):
    cfg = RUNS[rid]
    if cfg['kind'] == 'yolo':
        mm = YOLO(ckpt_path(rid, seed)); params = sum(p.numel() for p in mm.model.parameters()) / 1e6
        run1 = lambda f: mm.predict(f, imgsz=cfg['imgsz'], device='cpu', verbose=False)
        data = files[:warmup + n]
    else:
        mm = copy.deepcopy(load_model(rid, seed)).to('cpu').eval(); params = sum(p.numel() for p in mm.parameters()) / 1e6
        data = [to_tensor(Image.open(f).convert('RGB')) for f in files[:warmup + n]]
        def run1(x):
            with torch.no_grad(): return mm([x])
    for x in data[:warmup]: run1(x)
    ts = []
    for x in data[warmup:warmup + n]:
        t = time.perf_counter(); run1(x); ts.append((time.perf_counter() - t) * 1000)
    return float(np.mean(ts)), float(np.std(ts)), params

def eval_run(rid, seed, split, lat=False):
    """Hasil di-cache per (run, split): set uji hanya dihitung SEKALI."""
    name = run_name(rid, seed); fp = f'{OUT}/results/{name}_{split}_{RES_TAG}.json'
    if os.path.exists(fp): return json.load(open(fp))
    files, gts = SPLITS[split]
    model = load_model(rid, seed); preds = predict(rid, model, files)
    row = dict(run=name, rid=rid, seed=seed, split=split, **eval_map(preds, gts))
    if split == 'val':
        sweep = {t: count_metrics(preds, gts, t) for t in THR_GRID}
        thr = min(sweep, key=lambda t: sweep[t]['RBC']['MAE'])
    else:
        thr = json.load(open(f'{OUT}/results/{name}_val_{RES_TAG}.json'))['count_thr']   # ambang dari VALIDASI
    cm = count_metrics(preds, gts, thr)
    row.update(count_thr=thr, MAE_WBC=cm['WBC']['MAE'], MAE_RBC=cm['RBC']['MAE'], MAE_Platelets=cm['Platelets']['MAE'],
               MAPE_RBC=cm['RBC']['MAPE'], MAPE_Platelets=cm['Platelets']['MAPE'])
    if lat:
        l, s, p = latency_cpu(rid, seed, SPLITS['val'][0]); row.update(latency_cpu_ms=l, latency_std=s, params_M=p)
    gh = LEGACY_GPU_H.get(name)
    mp = f'{OUT}/runs/{name}/done.json'
    if os.path.exists(mp): gh = json.load(open(mp)).get('gpu_hours')
    row['gpu_hours'] = gh
    json.dump(row, open(fp, 'w'), indent=1)
    del model; torch.cuda.empty_cache()
    return row

## 3. Fungsi pelatihan

In [ ]:
def train_yolo(rid, seed):
    cfg = RUNS[rid]; name = run_name(rid, seed); t0 = time.time()
    kw = dict(freeze=cfg['freeze']) if cfg['freeze'] else {}
    YOLO(cfg['w']).train(data=YAML, imgsz=cfg['imgsz'], epochs=50, patience=15, batch=16, seed=seed, deterministic=True,
        hsv_h=0.0, hsv_s=0.2, hsv_v=0.2, flipud=0.5, fliplr=0.5, project=f'{OUT}/runs', name=name, exist_ok=True,
        workers=2, device=0, **kw)
    return (time.time() - t0) / 3600

def train_frcnn(rid, seed, EPOCHS=30, BATCH=4, LR=0.01, PATIENCE=8):
    cfg = RUNS[rid]; name = run_name(rid, seed); os.makedirs(f'{OUT}/runs/{name}', exist_ok=True)
    set_seed(seed); model = build_model(cfg['arch'], cfg['imgsz']).to(DEV)
    g = torch.Generator().manual_seed(seed)
    def wif(_): np.random.seed(seed); random.seed(seed)
    dl = torch.utils.data.DataLoader(TXL('train', True), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2, generator=g, worker_init_fn=wif)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.1); scaler = torch.amp.GradScaler('cuda')
    vf, vg = SPLITS['val']; best, bad, hist, t0 = -1, 0, [], time.time()
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for imgs, tg in dl:
            imgs = [i.to(DEV) for i in imgs]; tg = [{k: v.to(DEV) for k, v in t.items()} for t in tg]
            with torch.autocast('cuda'): loss = sum(model(imgs, tg).values())
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        sched.step()
        if ep % 2 == 1 or ep == EPOCHS - 1:
            r = eval_map(predict_torch(model, vf), vg); hist.append(dict(epoch=ep + 1, loss=tot / len(dl), mAP50=r['mAP50'], mAP50_95=r['mAP50_95'])); print(name, hist[-1])
            if r['mAP50_95'] > best: best, bad = r['mAP50_95'], 0; torch.save(model.state_dict(), ckpt_path(rid, seed))
            else:
                bad += 2
                if bad >= PATIENCE: print('early stopping'); break
    pd.DataFrame(hist).to_csv(f'{OUT}/runs/{name}/history.csv', index=False)
    del model; torch.cuda.empty_cache()
    return (time.time() - t0) / 3600

def train_run(rid, seed):
    name = run_name(rid, seed)
    if is_done(rid, seed): print('lewati (sudah selesai):', name); return
    print('=== LATIH', name, '===')
    try:
        set_seed(seed)
        gh = train_yolo(rid, seed) if RUNS[rid]['kind'] == 'yolo' else train_frcnn(rid, seed)
        os.makedirs(f'{OUT}/runs/{name}', exist_ok=True)
        json.dump(dict(gpu_hours=gh), open(f'{OUT}/runs/{name}/done.json', 'w'))
        print(f'selesai {name}: {gh:.2f} jam')
    except Exception:
        print('GAGAL', name); traceback.print_exc()
        torch.cuda.empty_cache()

## 4. Fase A · latih R4, R5, R2, R6 (seed 42)

In [ ]:
for rid in ['R1', 'R4', 'R5', 'R2', 'R6']:      # R1 ikut dilatih ulang bila belum ada
    train_run(rid, 42)

## 5. Evaluasi semua run (seed 42) pada VALIDASI, dengan fungsi dan grid ambang yang sama

In [ ]:
rows = []
for rid in ['R1', 'R2', 'R3', 'R4', 'R5', 'R6']:
    if os.path.exists(ckpt_path(rid, 42)) and is_done(rid, 42):
        rows.append(eval_run(rid, 42, 'val', lat=True))
    else: print('belum ada:', run_name(rid, 42))
VAL42 = pd.DataFrame(rows)
cols = ['rid', 'mAP50', 'mAP50_95', 'AP_small', 'AP_WBC', 'AP_RBC', 'AP_Platelets', 'count_thr', 'MAE_RBC', 'MAPE_RBC', 'MAE_Platelets', 'latency_cpu_ms', 'params_M', 'gpu_hours']
TAB3 = VAL42[cols].round(3); TAB3.to_csv(f'{OUT}/Tabel3_val_seed42.csv', index=False)
if (VAL42.count_thr == max(THR_GRID)).any(): print('PERINGATAN: ambang terbaik ada di tepi grid; pertimbangkan memperluas grid.')
TAB3

## 6. Pilih dua konfigurasi terbaik (satu per paradigma) → R7

In [ ]:
def best_of(kind):
    sub = VAL42[VAL42.rid.map(lambda r: RUNS[r]['kind'] == kind)]
    return sub.sort_values('mAP50_95', ascending=False).iloc[0].rid
FINALS = [best_of('yolo'), best_of('frcnn')]
print('Konfigurasi final (pilihan di validasi, mAP@[.5:.95]):', FINALS, [run_name(r, 42) for r in FINALS])

## 7. Fase B · seed 1 dan 2 untuk dua konfigurasi final (R7)

In [ ]:
SEEDS_EXTRA = [1, 2]      # kalau waktu mepet: ganti [1] atau [] (catat di laporan)
for rid in FINALS:
    for seed in SEEDS_EXTRA:
        train_run(rid, seed)
rows = []
for rid in FINALS:
    for seed in [42] + SEEDS_EXTRA:
        if is_done(rid, seed): rows.append(eval_run(rid, seed, 'val', lat=(seed == 42)))
R7 = pd.DataFrame(rows)
def msd(df, c): return f"{df[c].mean():.3f} ± {df[c].std(ddof=1):.3f}" if len(df) > 1 else f"{df[c].mean():.3f}"
tab7 = pd.DataFrame([dict(config=r, run=RUNS[r]['name'].replace('_s{seed}', ''), n_seed=len(R7[R7.rid == r]),
        mAP50=msd(R7[R7.rid == r], 'mAP50'), mAP50_95=msd(R7[R7.rid == r], 'mAP50_95'), AP_Platelets=msd(R7[R7.rid == r], 'AP_Platelets'),
        MAE_RBC=msd(R7[R7.rid == r], 'MAE_RBC'), MAPE_RBC=msd(R7[R7.rid == r], 'MAPE_RBC'),
        latency_ms_seed42=round(float(R7[(R7.rid == r) & (R7.seed == 42)].latency_cpu_ms.iloc[0]), 1)) for r in FINALS])
tab7.to_csv(f'{OUT}/Tabel_R7_val_3seed.csv', index=False); tab7

## 8. R8 · set UJI (sekali), konfigurasi final × 3 seed

In [ ]:
rows = [eval_run(rid, seed, 'test') for rid in FINALS for seed in [42] + SEEDS_EXTRA if is_done(rid, seed)]
R8 = pd.DataFrame(rows)
tab8 = pd.DataFrame([dict(config=r, n_seed=len(R8[R8.rid == r]), mAP50=msd(R8[R8.rid == r], 'mAP50'), mAP50_95=msd(R8[R8.rid == r], 'mAP50_95'),
        AP_Platelets=msd(R8[R8.rid == r], 'AP_Platelets'), MAE_RBC=msd(R8[R8.rid == r], 'MAE_RBC'), MAPE_RBC=msd(R8[R8.rid == r], 'MAPE_RBC')) for r in FINALS])
tab8.to_csv(f'{OUT}/Tabel_R8_test_3seed.csv', index=False); tab8

## 9. Uji hipotesis
H: YOLOv8n mencapai MAE hitung RBC dalam selisih ≤ 10% dari Faster R-CNN ResNet-50-FPN dengan latensi CPU ≥ 5× lebih cepat.

In [ ]:
def hyp(a, b, label):
    ra, rb = VAL42[VAL42.rid == a].iloc[0], VAL42[VAL42.rid == b].iloc[0]
    diff = (ra.MAE_RBC - rb.MAE_RBC) / rb.MAE_RBC * 100; speed = rb.latency_cpu_ms / ra.latency_cpu_ms
    ok = (diff <= 10) and (speed >= 5)
    print(f'{label}: MAE RBC {a}={ra.MAE_RBC:.3f} vs {b}={rb.MAE_RBC:.3f} -> selisih {diff:+.1f}% | percepatan {speed:.1f}x | hipotesis {"DIDUKUNG" if ok else "TIDAK didukung"}')
if {'R1', 'R5'} <= set(VAL42.rid): hyp('R1', 'R5', 'Validasi seed 42')

## 10. Galeri kesalahan (set uji) dan peta aktivasi EigenCAM
Dipakai model YOLO final seed 42. **Kategori kesalahan (sel bertumpuk, platelet kecil, dst.) kamu isi manual** dengan melihat galeri; file CSV memuat jumlah FP/FN per citra.

In [ ]:
def iou_mat(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)

def match_counts(pb, pl, ps, gb, gl, thr, iou_thr=0.5):
    """Hitung TP/FP/FN per kelas (greedy menurut skor)."""
    res = {}
    for c, n in enumerate(NAMES):
        m = (pl == c) & (ps >= thr); p = pb[m]; s = ps[m]; g = gb[gl == c]
        p = p[np.argsort(-s)]; tp = 0
        if len(p) and len(g):
            M = iou_mat(p, g); used = set()
            for i in range(len(p)):
                j = [k for k in np.argsort(-M[i]) if k not in used and M[i, k] >= iou_thr][:1]
                if j: used.add(j[0]); tp += 1
        res[n] = dict(TP=tp, FP=len(p) - tp, FN=len(g) - tp)
    return res

def eigencam(yolo, path, imgsz=640):
    feats = {}
    h = yolo.model.model[9].register_forward_hook(lambda m, i, o: feats.__setitem__('a', o.detach().float().cpu()[0]))
    yolo.predict(path, imgsz=imgsz, verbose=False, device=0); h.remove()
    a = feats['a'].numpy(); Cc, Hh, Ww = a.shape
    X = a.reshape(Cc, -1).T; X = X - X.mean(0)
    _, _, VT = np.linalg.svd(X, full_matrices=False); proj = (X @ VT[0]).reshape(Hh, Ww)
    if np.corrcoef(proj.ravel(), np.abs(a).mean(0).ravel())[0, 1] < 0: proj = -proj
    proj = np.maximum(proj, 0); proj = (proj - proj.min()) / (proj.max() - proj.min() + 1e-8)
    W, H = Image.open(path).size; sc = imgsz / max(W, H); nw, nh = round(W * sc), round(H * sc)
    big = np.array(Image.fromarray(proj.astype(np.float32), mode='F').resize((Ww * 32, Hh * 32), Image.BILINEAR))
    top, left = (Hh * 32 - nh) // 2, (Ww * 32 - nw) // 2
    crop = big[top:top + nh, left:left + nw]
    return np.array(Image.fromarray(crop.astype(np.float32), mode='F').resize((W, H), Image.BILINEAR))

In [ ]:
FY = FINALS[0]; tf, tg = SPLITS['test']
ym = load_model(FY, 42); tp_ = predict(FY, ym, tf)
thr = json.load(open(f'{OUT}/results/{run_name(FY, 42)}_val_{RES_TAG}.json'))['count_thr']
recs = []
for f, p, g in zip(tf, tp_, tg):
    mc = match_counts(p['boxes'].numpy(), p['labels'].numpy(), p['scores'].numpy(), g['boxes'].numpy(), g['labels'].numpy(), thr)
    recs.append(dict(img=os.path.basename(f), path=f, n_gt=len(g['labels']), FP=sum(v['FP'] for v in mc.values()), FN=sum(v['FN'] for v in mc.values()),
                     FP_plt=mc['Platelets']['FP'], FN_plt=mc['Platelets']['FN'], FN_rbc=mc['RBC']['FN'], FP_rbc=mc['RBC']['FP']))
ER = pd.DataFrame(recs); ER['err'] = ER.FP + ER.FN
ER.drop(columns='path').sort_values('err', ascending=False).to_csv(f'{OUT}/error_per_image_test.csv', index=False)
worst = ER.sort_values('err', ascending=False).head(8)
good = ER[ER.n_gt >= 10].sort_values('err').head(4)
def draw(path, p, g):
    im = Image.open(path).convert('RGB'); d = ImageDraw.Draw(im)
    for b in g['boxes'].numpy(): d.rectangle(list(b), outline='lime', width=2)
    for b, s in zip(p['boxes'].numpy(), p['scores'].numpy()):
        if s >= thr: d.rectangle(list(b), outline='red', width=1)
    return im
idx = {os.path.basename(f): i for i, f in enumerate(tf)}
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
for a, (_, r) in zip(ax.ravel(), worst.iterrows()):
    i = idx[r.img]; a.imshow(draw(r.path, tp_[i], tg[i])); a.set_title(f'{r.img[:18]}  FP={r.FP} FN={r.FN}', fontsize=9); a.axis('off')
plt.suptitle(f'Galeri kesalahan terbesar (hijau = label, merah = prediksi, ambang {thr})'); plt.tight_layout()
plt.savefig(f'{OUT}/figures/galeri_kesalahan.png', dpi=130); plt.show()

In [ ]:
# EigenCAM: 4 kasus benar (kiri) dan 4 kasus salah (kanan)
sel = [('benar', r) for _, r in good.iterrows()] + [('salah', r) for _, r in worst.head(4).iterrows()]
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
for a, (lab, r) in zip(ax.ravel(), sel):
    img = np.array(Image.open(r.path).convert('RGB')) / 255.0; hm = eigencam(ym, r.path, RUNS[FY]['imgsz'])
    a.imshow(img); a.imshow(hm, cmap='jet', alpha=0.45); a.set_title(f'{lab}: {r.img[:18]} (FP={r.FP}, FN={r.FN})', fontsize=9); a.axis('off')
plt.suptitle('EigenCAM (komponen utama aktivasi lapisan SPPF); bukan Grad-CAM berbasis gradien'); plt.tight_layout()
plt.savefig(f'{OUT}/figures/eigencam.png', dpi=130); plt.show()

## 11. Ringkasan untuk laporan

In [ ]:
print('Tabel 3 (val, seed 42):'); display(TAB3)
print('R7 (val, 3 seed):'); display(tab7)
print('R8 (uji, 3 seed):'); display(tab8)
print('Semua file tersimpan di', OUT)